# Tech Challenge - Fase 1 (IADT)

## Notebook principal - Trabalho obrigatorio

Este notebook concentra o fluxo principal do trabalho com **dados estruturados** do Breast Cancer Wisconsin. Aqui ficam as partes obrigatorias da entrega:

- contexto e problema
- carga e exploracao dos dados
- pre-processamento
- treino de varios modelos
- validacao cruzada
- avaliacao com accuracy, recall e F1
- ROC e matrizes de confusao
- explicabilidade
- ajuste de limiar orientado a recall

Os extras (CNN, radiomica e estudo multimodal) ficam no notebook `02_estudo_integrado.ipynb`.

## 1. Setup

Adicionamos `src/` ao path para importar o pacote local sem precisar instalar o projeto em modo editavel.

In [ ]:
%matplotlib inline
import os, sys
sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve, roc_auc_score
from sklearn.model_selection import cross_validate

from techchallenge import config
from techchallenge.data import estruturado as dados
from techchallenge.models import estruturado as modelos_estr
from techchallenge.evaluation import metrics, explicabilidade, limiar

sns.set_theme(style="whitegrid")
OUT = config.garantir_outputs()
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
config.PROJETO_DIR, OUT

## 2. Problema e base escolhida

A tarefa e classificar tumores de mama em **maligno** ou **benigno** a partir de caracteristicas numericas extraidas de exames. O dataset principal do trabalho e o **Breast Cancer Wisconsin**, com 569 casos e 30 features.

In [ ]:
df = dados.carregar_dados()
print("Formato:", df.shape)
df.head()

## 3. Exploracao de dados (EDA)

Primeiro verificamos distribuicao da classe alvo, estatisticas descritivas e correlacoes mais importantes com malignidade.

In [ ]:
counts = df["diagnosis"].value_counts().rename(index={"B": "Benigno", "M": "Maligno"})
display(counts.to_frame("quantidade"))

plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="diagnosis", order=["B", "M"], palette=["#4c9f70", "#d1495b"])
plt.title("Distribuicao do diagnostico")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.show()

In [ ]:
df.describe().T.head(10)

In [ ]:
df_corr = df.copy()
df_corr["diagnosis"] = df_corr["diagnosis"].map({"M": 1, "B": 0})
corr = df_corr.corr(numeric_only=True)["diagnosis"].drop("diagnosis").sort_values(ascending=False)
print("Top 10 features mais correlacionadas com malignidade:")
display(corr.head(10).round(3).to_frame("correlacao"))

plt.figure(figsize=(8, 10))
corr.head(15).sort_values().plot(kind="barh", color="#d1495b")
plt.title("Top correlacoes com malignidade")
plt.xlabel("Correlacao de Pearson")
plt.show()

In [ ]:
top_cols = corr.head(10).index.tolist()
plt.figure(figsize=(10, 8))
sns.heatmap(df_corr[top_cols + ["diagnosis"]].corr(), cmap="coolwarm", center=0)
plt.title("Heatmap das principais features")
plt.show()

## 4. Pre-processamento

O pacote faz internamente:

- remocao de colunas nao informativas
- encoding do alvo (`M=1`, `B=0`)
- split estratificado treino/teste
- padronizacao dentro de `Pipeline`, evitando data leakage

In [ ]:
X_train, X_test, y_train, y_test = dados.preparar_treino_teste(df)
print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("Proporcao no treino:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Proporcao no teste :", y_test.value_counts(normalize=True).round(3).to_dict())

## 5. Modelagem

Aqui treinamos os **6 modelos de base** do projeto e o **Stacking**, que hoje e o melhor modelo geral.

In [ ]:
modelos = modelos_estr.construir_modelos()
modelos["Stacking"] = modelos_estr.construir_stacking(modelos_estr.construir_modelos())

resultados = []
for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    linha = {"Modelo": nome, **metrics.metricas(y_test, y_pred)}
    cv = cross_validate(modelo, X_train, y_train, cv=5, scoring=["accuracy", "recall"], n_jobs=-1)
    linha["CV Acc (media)"] = cv["test_accuracy"].mean()
    linha["CV Recall (media)"] = cv["test_recall"].mean()
    resultados.append(linha)

res_df = pd.DataFrame(resultados).set_index("Modelo").round(4).sort_values(
    by=["Recall (maligno)", "Accuracy"], ascending=False
)
res_df

## 6. Avaliacao

A metrica prioritaria e o **recall da classe maligna**, porque o erro mais grave e o falso negativo.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.ravel()

for ax, (nome, modelo) in zip(axes, modelos.items()):
    cm = confusion_matrix(y_test, modelo.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=["Benigno", "Maligno"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(nome)

for ax in axes[len(modelos):]:
    ax.axis("off")

plt.suptitle("Matrizes de confusao - modelos do trabalho", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))
for nome, modelo in modelos.items():
    if hasattr(modelo, "predict_proba"):
        prob = modelo.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc = roc_auc_score(y_test, prob)
        lw = 2.5 if nome == "Stacking" else 1.3
        plt.plot(fpr, tpr, lw=lw, label=f"{nome} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("Falso positivo (1 - especificidade)")
plt.ylabel("Verdadeiro positivo (recall)")
plt.title("Curvas ROC - modulo estruturado")
plt.legend(loc="lower right", fontsize=8)
plt.show()

## 7. Explicabilidade

Geramos os graficos de interpretabilidade com as funcoes do pacote e exibimos no notebook.

In [ ]:
explicabilidade.feature_importance_logistica(
    modelos["Regressao Logistica"],
    X_train.columns,
    OUT / "estruturado_feature_importance.png",
)
display(Image(str(OUT / "estruturado_feature_importance.png")))

In [ ]:
explicabilidade.shap_arvore(
    modelos["Arvore de Decisao"],
    X_test,
    X_train.columns,
    OUT / "estruturado_shap.png",
)
display(Image(str(OUT / "estruturado_shap.png")))

## 8. Ajuste de limiar para prioridade clinica

O melhor modelo global e o **Stacking**. Aqui ajustamos o limiar para priorizar recall e reduzir falsos negativos.

In [ ]:
prob = modelos["Stacking"].predict_proba(X_test)[:, 1]
tab = limiar.tabela_limiares(y_test, prob)
t_recall = limiar.escolher_limiar_por_recall(y_test, prob, recall_alvo=1.0)
limiar.plot_trade_off(tab, OUT / "estruturado_limiar_tradeoff.png", t_recall)
limiar.comparar_matrizes(y_test, prob, 0.5, t_recall, OUT / "estruturado_limiar_matrizes.png")

def resumo_limiar(t):
    pred = (prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    rec = tp / (tp + fn)
    return {"limiar": t, "FN": int(fn), "FP": int(fp), "Recall": round(rec, 4)}

pd.DataFrame([resumo_limiar(0.5), resumo_limiar(t_recall)])

In [ ]:
display(Image(str(OUT / "estruturado_limiar_tradeoff.png")))
display(Image(str(OUT / "estruturado_limiar_matrizes.png")))

## 9. Conclusao

O notebook principal mostra que o modulo estruturado cumpre os requisitos obrigatorios do trabalho. Hoje, o melhor resultado geral e do **Stacking**, com recall alto para a classe maligna e bom desempenho global.

Ao mesmo tempo, a **Regressao Logistica** continua muito valiosa por ser mais interpretavel. Em contexto clinico, a decisao final deve sempre permanecer com o medico.

Os estudos extras de imagem, radiomica, CNN e multimodalidade ficam no notebook `02_estudo_integrado.ipynb`.